# GNN edge classifier
Laundering detection on IBM AML HI-Small with the cleaned data, split and features (stage A). Accounts are nodes and
payments are directed edges.

Each payment is classified from its own features and its sampled neighbourhood.

## 1. Setup


In [21]:
import os, sys, glob, json, shutil, subprocess, zipfile
from google.colab import drive
drive.mount('/content/drive')

CODE_DIR = '/content/drive/MyDrive/bt4012_gnn_team/repo'
DATA_DRIVE = '/content/drive/MyDrive/bt4012_gnn/data_team'
OUT_DRIVE = '/content/drive/MyDrive/bt4012_gnn_team/outputs'

RUN_LADDER, RUN_SEEDS, RUN_OUTPUTS = True, True, True

REPO = '/content/repo'
if os.path.islink(f'{REPO}/outputs'):
    os.unlink(f'{REPO}/outputs')
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(CODE_DIR, REPO, ignore=shutil.ignore_patterns('outputs', '__pycache__', '.git'))
os.makedirs(OUT_DRIVE, exist_ok=True)
os.symlink(OUT_DRIVE, f'{REPO}/outputs')
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
import torch, psutil
ram = psutil.virtual_memory().total / 2**30
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'} | RAM: {ram:.0f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: none | RAM: 51 GB


In [22]:
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)
for f in ('validation.csv', 'test.csv', 'pattern_labels.csv'):
    if not os.path.exists(f'{DATA}/{f}'):
        shutil.copy2(f'{DATA_DRIVE}/{f}', f'{DATA}/{f}')
if not os.path.exists(f'{DATA}/train.csv'):
    if os.path.exists(f'{DATA_DRIVE}/train.csv'):
        shutil.copy2(f'{DATA_DRIVE}/train.csv', f'{DATA}/train.csv')
    else:
        with zipfile.ZipFile(f'{DATA_DRIVE}/train.csv.zip') as z:
            z.extract('train.csv', DATA)
print(sorted(os.listdir(DATA)))

# team config, pointed at this session's data folder
import yaml
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['gnn']['data_dir'] = DATA
yaml.safe_dump(cfg, open('configs/colab.yaml', 'w'), sort_keys=False)
G = cfg['gnn']
print('runs:', [r['name'] for r in G['runs']])

['pattern_labels.csv', 'test.csv', 'train.csv', 'validation.csv']
runs: ['gin_plain', 'multigin', 'multigin_hist', 'multigin_causal', 'multigin_causal_mega', 'pna_final']


In [23]:
def stage(*args, log='stage04'):
    """Runs scripts/04_train_gnn.py, streaming to the screen and to outputs/gnn/logs/<log>.log."""
    os.makedirs('outputs/gnn/logs', exist_ok=True)
    cmd = [sys.executable, '-W', 'ignore', 'scripts/04_train_gnn.py', '--config', 'configs/colab.yaml', *args]
    env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    with open(f'outputs/gnn/logs/{log}.log', 'a') as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in p.stdout:
            print(line, end='')
            fh.write(line)
        if p.wait():
            raise RuntimeError(f'stage 04 failed; see outputs/gnn/logs/{log}.log')

def finished(name):
    return os.path.exists(f'outputs/gnn/runs/{name}/metrics.json')

def need_ram(names):
    left = [n for n in names if not finished(n)]
    if left and ram < 25:
        raise SystemExit(f'{len(left)} runs left but only {ram:.0f} GB RAM: switch to an L4 High-RAM runtime')
    return left

## 2. Ablation ladder (seed 0)
We compare runs by validation AUPRC. Runs 1 to 3 let the model see the whole graph, including later payments, as the
Multi-GNN benchmark does, so they are only for comparison with published results. From run 4 on, nothing later is
ever used: each minute's payments are scored at the end of that minute, using only payments up to then.

1. `gin_plain`: a basic GNN (GIN) whose inputs are only the payments' own details (no account history)
2. `multigin`: adds IBM's Multi-GNN changes. Messages flow both ways along a payment, each account tells its links
   apart (ports), the two accounts of the payment being scored are marked (ego IDs), and payment features are updated
   at every layer
3. `multigin_hist`: adds the 32 account-history features from stage A
4. `multigin_causal`: the model no longer sees payments made after the one it is scoring
5. `multigin_causal_mega`: combines repeated payments between the same two accounts before passing messages (MEGA)
6. `pna_final`: switches to a PNA GNN, which summarises each account's neighbours in several ways (mean, max, min,
   spread). This is the final model

In [24]:
LADDER = [f"{r['name']}_seed0" for r in G['runs']]
if RUN_LADDER and need_ram(LADDER):
    stage('--seeds', '0', log='ladder')
print({n: finished(n) for n in LADDER})

{'gin_plain_seed0': True, 'multigin_seed0': True, 'multigin_hist_seed0': True, 'multigin_causal_seed0': True, 'multigin_causal_mega_seed0': True, 'pna_final_seed0': True}


## 3. Final model: seeds 1 and 2
The report quotes the final model as mean ± sd over seeds 0, 1 and 2.

In [25]:
FINAL = G['final']
SEEDS = [f'{FINAL}_seed{s}' for s in (0, 1, 2)]
if RUN_SEEDS and need_ram(SEEDS[1:]):
    stage('--runs', FINAL, '--seeds', '1', '2', log='seeds')
print({n: finished(n) for n in SEEDS})

{'pna_final_seed0': True, 'pna_final_seed1': True, 'pna_final_seed2': True}


## 4. Predictions file and explanations
* `outputs/predictions/predictions_gnn.parquet` (`edge_id`, `split`, `score`): scores from the final seed with the best validation AUPRC, used by the shared evaluation step (stage 05) to compare the GNN with the baseline.
* `outputs/agent/alerts_explained.json`: the 100 highest-scoring test payments (a small set for the agent, not all alerts), for the investigation agent (stage 06). For each one, it lists the five details of the payment and the five nearby payments that most influenced its score.

In [26]:
if RUN_OUTPUTS and all(finished(n) for n in SEEDS):
    stage('--publish', log='outputs')
    if not os.path.exists('outputs/agent/alerts_explained.json'):
        stage('--explain', '--pattern_labels', f'{DATA}/pattern_labels.csv', log='outputs')
print(os.listdir('outputs/predictions') if os.path.exists('outputs/predictions') else 'no predictions yet')

wrote outputs/predictions/predictions_gnn.parquet from pna_final_seed0 (1523381 rows)
['predictions_gnn.parquet', 'predictions_gnn_metrics.json']


## 5. Report tables

In [27]:
import pandas as pd, numpy as np
pd.set_option('display.width', 200)
res = pd.read_csv('outputs/gnn/results.csv').set_index('name')
lad = res.loc[[n for n in LADDER if n in res.index]]
t = lad[['val_auprc', 'val_auprc_half1', 'val_auprc_half2', 'test_auprc', 'test_f1', 'minutes']].copy()
t['gain_vs_previous'] = t['val_auprc'].diff()
gains = lad[['val_auprc_half1', 'val_auprc_half2']].diff()
t['wins_both_halves_by_0.02'] = (gains > 0.02).all(axis=1)
print('Ablation ladder (seed 0)')
display(t.round(4))
fin = res.loc[[n for n in SEEDS if n in res.index]]
cols = ['val_auprc', 'test_auprc', 'test_roc_auc', 'test_f1', 'test_recall@1000', 'test_recall@5000']
print(f'Final model, {len(fin)} seeds')
display(fin[cols].round(4))
display(fin[cols].agg(['mean', 'std']).T.round(4))

Ablation ladder (seed 0)


,val_auprc,val_auprc_half1,val_auprc_half2,test_auprc,test_f1,minutes,gain_vs_previous,wins_both_halves_by_0.02
name,,,,,,,,
gin_plain_seed0,0.4074,0.3983,0.4180,0.5582,0.5812,66.9,NaN,False
multigin_seed0,0.5304,0.5230,0.5406,0.7380,0.7403,93.6,0.1230,True
multigin_hist_seed0,0.5972,0.6030,0.5942,0.7861,0.7976,96.5,0.0668,True
multigin_causal_seed0,0.5348,0.5440,0.5305,0.6858,0.7007,69.4,-0.0624,False
multigin_causal_mega_seed0,0.5714,0.5683,0.5740,0.7492,0.7402,69.0,0.0365,True
pna_final_seed0,0.6107,0.6176,0.6079,0.7809,0.7876,131.4,0.0394,True


Final model, 3 seeds


,val_auprc,test_auprc,test_roc_auc,test_f1,test_recall@1000,test_recall@5000
name,,,,,,
pna_final_seed0,0.6107,0.7809,0.9926,0.7876,0.6060,0.8725
pna_final_seed1,0.6021,0.7701,0.9936,0.7817,0.5996,0.8732
pna_final_seed2,0.5873,0.7446,0.9914,0.7663,0.5842,0.8463


,mean,std
val_auprc,0.6000,0.0119
test_auprc,0.7652,0.0186
test_roc_auc,0.9925,0.0011
test_f1,0.7785,0.0110
test_recall@1000,0.5966,0.0112
test_recall@5000,0.8640,0.0154


In [28]:
# per-typology results on test
best = fin['val_auprc'].idxmax()
thr = json.load(open(f'outputs/gnn/runs/{best}/metrics.json'))['val']['threshold']
pred = pd.read_parquet(f'outputs/gnn/runs/{best}/predictions.parquet').query("split == 'test'")
lab = pd.read_csv(f'{DATA}/test.csv', usecols=['source_row', 'is_laundering']).rename(columns={'source_row': 'edge_id'})
pred = pred.merge(lab, on='edge_id')
pl = pd.read_csv(f'{DATA}/pattern_labels.csv')
p = pred.merge(pl[['source_row', 'pattern_id', 'typology']].rename(columns={'source_row': 'edge_id'}), on='edge_id', how='left')
p['typology'] = p['typology'].fillna('UNKNOWN_UNLISTED')
p = p[p.is_laundering == 1]
p['flagged'] = p['score'] >= thr
typ = p.groupby('typology').agg(transactions=('edge_id', 'size'), recall_at_threshold=('flagged', 'mean'))
att = p.dropna(subset=['pattern_id']).groupby(['typology', 'pattern_id'])['flagged'].max().groupby('typology')
typ['schemes'] = att.size()
typ['schemes_with_a_flag'] = att.mean()
print(f'{best}, test, threshold {thr:.3f} chosen on validation')
display(typ.round(3))

# review tiers: share of test laundering caught when analysts review the top x% of test payments
s = pred.sort_values('score', ascending=False)
tiers = {f'top {q}%': s['is_laundering'].iloc[:int(len(s) * q / 100)].sum() / s['is_laundering'].sum()
         for q in (0.1, 0.5, 1, 5, 10)}
display(pd.Series(tiers, name='recall').round(3).to_frame())

pna_final_seed0, test, threshold 0.891 chosen on validation


,transactions,recall_at_threshold,schemes,schemes_with_a_flag
typology,,,,
BIPARTITE,62,0.774,19.0,1.000
CYCLE,96,0.875,22.0,0.955
FAN-IN,122,0.951,19.0,1.000
FAN-OUT,129,0.969,21.0,1.000
GATHER-SCATTER,379,0.971,37.0,1.000
RANDOM,82,0.927,17.0,1.000
SCATTER-GATHER,235,0.974,22.0,1.000
STACK,114,0.833,24.0,1.000
UNKNOWN_UNLISTED,342,0.003,NaN,NaN


,recall
top 0.1%,0.466
top 0.5%,0.843
top 1%,0.892
top 5%,0.939
top 10%,0.974


In [29]:
# where the explanations put the weight (own payment / 1 hop / 2 hops), by outcome group
f = f'outputs/gnn/runs/{best}/explain_eval/explain_summary.csv'
display(pd.read_csv(f).round(3) if os.path.exists(f) else 'run section 4 first')

,group,n,mean_score,share_own_tx,share_1hop,share_2hop,median_context_edges
0,caught_pattern,200,0.993,0.469,0.462,0.068,452.5
1,missed_pattern,78,0.544,0.517,0.430,0.053,190.0
2,caught_nopattern,1,0.903,0.577,0.406,0.017,109.0
3,missed_nopattern,200,0.354,0.451,0.513,0.036,183.5
4,false_alarm_top,200,0.957,0.505,0.438,0.056,285.5
5,normal_random,200,0.008,0.343,0.631,0.025,297.0


## 6. Watchlist follow-up (post-processing for the agent)
- Once a case is confirmed, the sender goes on a watchlist and its later payments are raised for review from 24 h after the confirmation, whatever their GNN score.
- Busy accounts (more than 20 payments sent or received before the confirmation) are left out and should go to an account review instead.
- Labels stand in for confirmed cases, and a case counts as confirmed at the time of its payment. `gnn_confirmed` watches accounts whose GNN alerts are truly laundering (we assume a reviewer confirms them). `+past_cases` also watches accounts the bank had already caught laundering before, in the training data.
- Each row of the table is one combination of three choices: who is watched (`sender`: only the account that sent the confirmed payment; `either`: the sender and the receiver), the busy-account cap (`20` or `none`), and which confirmed cases are used. We recommend `sender`, cap 20, `+past_cases`; the other rows show that watching both accounts or removing the cap adds many more alerts for few extra catches.

In [30]:
if all(finished(n) for n in SEEDS):
    if not os.path.exists('outputs/gnn/watchlist_eval.csv'):
        stage('--watchlist', '--pattern_labels', f'{DATA}/pattern_labels.csv', log='watchlist')
    wl = pd.read_csv('outputs/gnn/watchlist_eval.csv')
    wl = wl[wl['confirmations'].isin(['gnn_confirmed', 'gnn_confirmed+past_cases'])]
    show = ['key', 'confirmations', 'max_prior_payments', 'extra_alerts', 'extra_alerts_pct_of_payments',
            'extra_alert_precision', 'unpatterned_recall_gnn', 'unpatterned_recall_gnn+watchlist',
            'unpatterned_extra_caught', 'all_recall_gnn', 'all_recall_gnn+watchlist']
    display(wl[show].round(4))

,key,confirmations,max_prior_payments,extra_alerts,extra_alerts_pct_of_payments,extra_alert_precision,unpatterned_recall_gnn,unpatterned_recall_gnn+watchlist,unpatterned_extra_caught,all_recall_gnn,all_recall_gnn+watchlist
0,sender,gnn_confirmed,20,144,0.0189,0.0208,0.0029,0.0058,1,0.7316,0.7335
1,sender,gnn_confirmed,none,1077,0.1414,0.0028,0.0029,0.0058,1,0.7316,0.7335
2,sender,gnn_confirmed+past_cases,20,1613,0.2118,0.0068,0.0029,0.0292,9,0.7316,0.7386
3,sender,gnn_confirmed+past_cases,none,76749,10.0769,0.0014,0.0029,0.3158,107,0.7316,0.8014
10,either,gnn_confirmed,20,454,0.0596,0.0154,0.0029,0.0058,1,0.7316,0.7361
11,either,gnn_confirmed,none,3169,0.4161,0.0032,0.0029,0.0058,1,0.7316,0.7380
12,either,gnn_confirmed+past_cases,20,9089,1.1934,0.0034,0.0029,0.0497,16,0.7316,0.7514
13,either,gnn_confirmed+past_cases,none,89012,11.6870,0.0015,0.0029,0.3304,112,0.7316,0.8155
